# Prework - Škálování dat: Normalizace a Standardizace (Heart Dataset)

Různé spojité numerické proměnné mají často velmi odlišná měřítka (např. věk `age` se pohybuje mezi 29 a 77, zatímco cholesterol `chol` dosahuje hodnot až 564). Modely citlivé na vzdálenosti (např. k-NN, SVM, lineární modely s regularizací, neuronové sítě) by bez škálování přisuzovaly neúměrně velkou váhu proměnným s většími číselnými hodnotami.

V tomto cvičení otestujeme dva základní přístupy ke škálování spojitých numerických dat:
1. **Normalizace (Min-Max Scaling)**: Převede hodnoty do fixního intervalu $[0, 1]$:
   $$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
2. **Standardizace (Z-score Standardization)**: Transformuje rozdělení tak, aby mělo průměr $\mu = 0$ a směrodatnou odchylku $\sigma = 1$:
   $$z = \frac{x - \mu}{\sigma}$$

*Poznámka: Binární dummy proměnné (hodnoty 0 a 1) vzniklé One-Hot kódováním v předchozím kroku se již neškálují.*

## 1. Načtení datasetu ze cvičení 'Processing categorical data'

In [1]:
import os
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

input_path = os.path.join("data", "heart_data_exercise_2.csv")
df = pd.read_csv(input_path)

print(f"Rozměry načtených dat: {df.shape[0]} řádků, {df.shape[1]} sloupců")
df.head()

Rozměry načtených dat: 303 řádků, 18 sloupců


,age,restbp,chol,maxhr,oldpeak,slope,ca,sex_male,chestpain_nonanginal,chestpain_nontypical,chestpain_typical,fbs_yes,restecg_1,restecg_2,exang_yes,thal_normal,thal_reversable,ahd_yes
0,63,145,233.0,150,2.3,3,0.0,1,0,0,1,1,0,1,0,0,0,0
1,67,160,286.0,108,1.5,2,3.0,1,0,0,0,0,0,1,1,1,0,1
2,67,120,229.0,129,2.6,2,2.0,1,0,0,0,0,0,1,1,0,1,1
3,37,130,240.0,187,3.5,3,0.0,1,1,0,0,0,0,0,0,0,1,0
4,41,130,204.0,172,1.4,1,0.0,0,0,1,0,0,0,1,0,0,0,0


### Identifikace spojitých numerických proměnných
Oddělíme spojité numerické proměnné od binárních dummy sloupců.

In [2]:
numeric_cols = ["age", "restbp", "chol", "maxhr", "oldpeak", "slope", "ca"]
dummy_cols = [col for col in df.columns if col not in numeric_cols]
ordered_cols = dummy_cols + numeric_cols

print("Spojité proměnné ke škálování:", numeric_cols)
print("Binární dummy sloupce:", dummy_cols)

print("\nZákladní statistiky před škálováním:")
df[numeric_cols].describe().T[["mean", "std", "min", "max"]].round(2)

Spojité proměnné ke škálování: ['age', 'restbp', 'chol', 'maxhr', 'oldpeak', 'slope', 'ca']
Binární dummy sloupce: ['sex_male', 'chestpain_nonanginal', 'chestpain_nontypical', 'chestpain_typical', 'fbs_yes', 'restecg_1', 'restecg_2', 'exang_yes', 'thal_normal', 'thal_reversable', 'ahd_yes']

Základní statistiky před škálováním:


,mean,std,min,max
age,54.44,9.04,29.0,77.0
restbp,131.69,17.60,94.0,200.0
chol,246.61,51.78,126.0,564.0
maxhr,149.61,22.88,71.0,202.0
oldpeak,1.04,1.16,0.0,6.2
slope,1.60,0.62,1.0,3.0
ca,0.66,0.93,0.0,3.0


## 2. Přístup 1: Normalizace dat (Min-Max Scaling)

Použijeme třídu `MinMaxScaler` ze Scikit-learn pro transformaci hodnot do intervalu $[0, 1]$ a uložíme výsledek do proměnné `heart_data_normalized`.

In [3]:
min_max = MinMaxScaler()

heart_data_normalized = df.copy()
heart_data_normalized[numeric_cols] = min_max.fit_transform(heart_data_normalized[numeric_cols])
heart_data_normalized = heart_data_normalized[ordered_cols]

print("Minima po normalizaci:")
print(heart_data_normalized[numeric_cols].min())
print("\nMaxima po normalizaci:")
print(heart_data_normalized[numeric_cols].max())

heart_data_normalized.head()

Minima po normalizaci:
age        0.0
restbp     0.0
chol       0.0
maxhr      0.0
oldpeak    0.0
slope      0.0
ca         0.0
dtype: float64

Maxima po normalizaci:
age        1.0
restbp     1.0
chol       1.0
maxhr      1.0
oldpeak    1.0
slope      1.0
ca         1.0
dtype: float64


,sex_male,chestpain_nonanginal,chestpain_nontypical,chestpain_typical,fbs_yes,restecg_1,restecg_2,exang_yes,thal_normal,thal_reversable,ahd_yes,age,restbp,chol,maxhr,oldpeak,slope,ca
0,1,0,0,1,1,0,1,0,0,0,0,0.708333,0.481132,0.244292,0.603053,0.370968,1.0,0.000000
1,1,0,0,0,0,0,1,1,1,0,1,0.791667,0.622642,0.365297,0.282443,0.241935,0.5,1.000000
2,1,0,0,0,0,0,1,1,0,1,1,0.791667,0.245283,0.235160,0.442748,0.419355,0.5,0.666667
3,1,1,0,0,0,0,0,0,0,1,0,0.166667,0.339623,0.260274,0.885496,0.564516,1.0,0.000000
4,0,0,1,0,0,0,1,0,0,0,0,0.250000,0.339623,0.178082,0.770992,0.225806,0.0,0.000000


## 3. Přístup 2: Standardizace dat (Z-score)

Použijeme třídu `StandardScaler` ze Scikit-learn pro standardizaci na $\mu = 0$ a $\sigma = 1$ a uložíme výsledek do proměnné `heart_data_standardized`.

In [4]:
std_scaler = StandardScaler()

heart_data_standardized = df.copy()
heart_data_standardized[numeric_cols] = std_scaler.fit_transform(heart_data_standardized[numeric_cols])
heart_data_standardized = heart_data_standardized[ordered_cols]

print("Průměry po standardizaci (očekáváno ~0.0):")
print(heart_data_standardized[numeric_cols].mean().round(4))
print("\nSměrodatné odchylky po standardizaci (očekáváno ~1.0):")
print(heart_data_standardized[numeric_cols].std().round(4))

heart_data_standardized.head()

Průměry po standardizaci (očekáváno ~0.0):
age       -0.0
restbp     0.0
chol       0.0
maxhr     -0.0
oldpeak    0.0
slope      0.0
ca        -0.0
dtype: float64

Směrodatné odchylky po standardizaci (očekáváno ~1.0):
age        1.0017
restbp     1.0017
chol       1.0017
maxhr      1.0017
oldpeak    1.0017
slope      1.0017
ca         1.0017
dtype: float64


,sex_male,chestpain_nonanginal,chestpain_nontypical,chestpain_typical,fbs_yes,restecg_1,restecg_2,exang_yes,thal_normal,thal_reversable,ahd_yes,age,restbp,chol,maxhr,oldpeak,slope,ca
0,1,0,0,1,1,0,1,0,0,0,0,0.948726,0.757525,-0.263364,0.017197,1.087338,2.274579,-0.711131
1,1,0,0,0,0,0,1,1,1,0,1,1.392002,1.611220,0.761937,-1.821905,0.397182,0.649113,2.504881
2,1,0,0,0,0,0,1,1,0,1,1,1.392002,-0.665300,-0.340745,-0.902354,1.346147,0.649113,1.432877
3,1,1,0,0,0,0,0,0,0,1,0,-1.932564,-0.096170,-0.127947,1.637359,2.122573,2.274579,-0.711131
4,0,0,1,0,0,0,1,0,0,0,0,-1.489288,-0.096170,-0.824378,0.980537,0.310912,-0.976352,-0.711131


## 4. Uložení obou datasetů do souborů .csv bez indexu

Oba datasety uložíme pomocí parametru `index=False`:
- `heart_data_normalized.csv`
- `heart_data_standardized.csv`

In [5]:
norm_path = os.path.join("data", "heart_data_normalized.csv")
std_path = os.path.join("data", "heart_data_standardized.csv")

heart_data_normalized.to_csv(norm_path, index=False)
heart_data_standardized.to_csv(std_path, index=False)

print(f"Normalizovaný dataset uložen do: '{norm_path}' ({heart_data_normalized.shape})")
print(f"Standardizovaný dataset uložen do: '{std_path}' ({heart_data_standardized.shape})")

Normalizovaný dataset uložen do: 'data\heart_data_normalized.csv' ((303, 18))
Standardizovaný dataset uložen do: 'data\heart_data_standardized.csv' ((303, 18))
